In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [9]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()

✅ Banco e tabela 'articles' prontos!


In [10]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [11]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

noticias = []
total_pages = 186

for page in range(1, total_pages + 1):
    if page == 1:
        url = "https://www.observacom.org/seccion/actualidad/"
    else:
        url = f"https://www.observacom.org/seccion/actualidad/page/{page}/"
    
    print(f"📄 Coletando página {page}/{total_pages}: {url}")
    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"❌ Erro ao acessar {url}: {e}")
        continue
        
    soup = BeautifulSoup(r.text, "html.parser")

    paineis = soup.find_all("article")
    if not paineis:
        paineis = [h2.parent for h2 in soup.find_all("h2") if h2.find("a")]

    if not paineis:
        print(f"⚠️ Nenhuma notícia encontrada na página {page}")
        continue

    print(f"   {len(paineis)} notícias encontradas")

    for item in paineis:
        titulo_tag = item.find(["h2", "h3"])
        if not titulo_tag:
            continue
            
        link_tag = titulo_tag.find("a")
        if not link_tag:
            continue
            
        link = link_tag.get("href")
        
        data_tag = item.find("time")
        
        author_tag = item.find(class_=re.compile("author"))
        author = author_tag.get_text(strip=True) if author_tag else "Observacom"

        data = {
            "title": titulo_tag.get_text(strip=True),
            "date": data_tag.get_text(strip=True) if data_tag else None,
            "author": author,
            "url": link,
            "source": "Observacom"
        }

        noticias.append(data)
        insert_article(**data)
        
    time.sleep(1) # pausa para não sobrecarregar o servidor

print(f"\n✅ Total coletado na Observacom nesta rodada: {len(noticias)} notícias")
df_observacom = pd.DataFrame(noticias)
if not df_observacom.empty:
    display(df_observacom.head())

📄 Coletando página 1/186: https://www.observacom.org/seccion/actualidad/
   12 notícias encontradas
📄 Coletando página 2/186: https://www.observacom.org/seccion/actualidad/page/2/
   12 notícias encontradas
📄 Coletando página 3/186: https://www.observacom.org/seccion/actualidad/page/3/
   12 notícias encontradas
📄 Coletando página 4/186: https://www.observacom.org/seccion/actualidad/page/4/
   12 notícias encontradas
📄 Coletando página 5/186: https://www.observacom.org/seccion/actualidad/page/5/
   12 notícias encontradas
📄 Coletando página 6/186: https://www.observacom.org/seccion/actualidad/page/6/
   12 notícias encontradas
📄 Coletando página 7/186: https://www.observacom.org/seccion/actualidad/page/7/
   12 notícias encontradas
📄 Coletando página 8/186: https://www.observacom.org/seccion/actualidad/page/8/
   12 notícias encontradas
📄 Coletando página 9/186: https://www.observacom.org/seccion/actualidad/page/9/
   12 notícias encontradas
📄 Coletando página 10/186: https://www.obser

,title,date,author,url,source
0,Dos fallos judiciales en EEUU atribuyen respon...,"27 marzo, 2026",Observacom,https://www.observacom.org/dos-fallos-judicial...,Observacom
1,La mitad de las noticias que recomienda Google...,"25 marzo, 2026",Observacom,https://www.observacom.org/la-mitad-de-las-not...,Observacom
2,Brasil actualiza su regulación de la inteligen...,"17 marzo, 2026",Observacom,https://www.observacom.org/brasil-actualiza-su...,Observacom
3,Prohibición de acceso a redes sociales: expert...,"12 marzo, 2026",Observacom,https://www.observacom.org/prohibicion-de-acce...,Observacom
4,Plataformas digitales redujeron sus acciones c...,"11 marzo, 2026",Observacom,https://www.observacom.org/plataformas-digital...,Observacom


In [12]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT * FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head(20))

📦 Total no banco: 2267 registros


,id,title,date,author,url,source
0,792,Nueva sentencia de la Corte Europea determina ...,"9 septiembre, 2021",Observacom,https://www.observacom.org/nueva-sentencia-de-...,Observacom
1,1100,Gigantes tecnológicos centralizan el poder de ...,"9 septiembre, 2020",Observacom,https://www.observacom.org/gigantes-tecnologic...,Observacom
2,1356,Lobby de grupos evangélicos para reformar ley ...,"9 septiembre, 2019",Observacom,https://www.observacom.org/lobby-de-grupos-eva...,Observacom
3,1960,AMARC señala falta de voluntad política para r...,"9 septiembre, 2016",Observacom,https://www.observacom.org/amarc-senala-falta-...,Observacom
4,236,"Meta, Google y TikTok concentran el 70% del tr...","9 octubre, 2024",Observacom,https://www.observacom.org/meta-google-y-tikto...,Observacom
5,371,EEUU: Plataformas de servicios audiovisuales e...,"9 octubre, 2023",Observacom,https://www.observacom.org/eeuu-plataformas-de...,Observacom
6,1066,Regulador audiovisual de Cataluña solicitó a T...,"9 octubre, 2020",Observacom,https://www.observacom.org/regulador-audiovisu...,Observacom
7,1067,El proyecto oficialista de Ley de Medios promu...,"9 octubre, 2020",Observacom,https://www.observacom.org/el-proyecto-oficial...,Observacom
8,1334,República Dominicana comenzará a cobrar impues...,"9 octubre, 2019",Observacom,https://www.observacom.org/republica-dominican...,Observacom
9,1546,Sociedad Civil chilena solicita participar en ...,"9 octubre, 2018",Observacom,https://www.observacom.org/sociedad-civil-chil...,Observacom


In [13]:
def plot_charts(df):
    if df.empty:
        print("❌ Sem dados para plotar")
        return

    # Top 15
    top15 = df.head(15).copy()
    top15["rank"] = range(1, len(top15) + 1)

    fig1 = px.bar(
        top15,
        x="rank",
        y="title",
        orientation="h",
        title="Top 15 Notícias – Internet Governance"
    )
    fig1.update_layout(height=600)
    fig1.show()

    # Fonte
    source_count = df["source"].value_counts().reset_index()
    source_count.columns = ["source", "count"]

    fig2 = px.pie(
        source_count,
        names="source",
        values="count",
        title="Distribuição por Fonte"
    )
    fig2.show()

    # Palavras
    text = " ".join(df["title"].astype(str)).lower()
    words = re.findall(r"\b\w{4,}\b", text)

    word_freq = (
        pd.Series(words)
        .value_counts()
        .head(20)
        .reset_index()
    )
    word_freq.columns = ["palavra", "freq"]

    fig3 = px.treemap(
        word_freq,
        path=["palavra"],
        values="freq",
        title="Palavras mais frequentes nos títulos"
    )
    fig3.show()

plot_charts(df_db)